# FaceProof 09 — aggregate + freeze results (Day 9)

Read the single `results.csv` and build the master tables for the report. **No new experiments**
after today. Saves markdown tables to `reports/` for direct paste into the write-up.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q pandas
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)

## 2. Load + master AUROC table (condition × generator)

In [ ]:
r = pd.read_csv(RESULTS_CSV)
r['seed'] = pd.to_numeric(r['seed'], errors='coerce')
r['value'] = pd.to_numeric(r['value'], errors='coerce')
r = r[r['corruption']=='none']
auc = r[(r['metric']=='AUROC') & (r['seed']==13)]
table = auc.pivot_table(index='condition', columns='test_gen', values='value', aggfunc='last')
print(table.round(4).to_string())
(REPORTS_ROOT/'table_auroc.md').write_text(table.round(4).to_markdown())

## 3. Generalization gap + calibration table

In [ ]:
if 'stylegan_indist' in table.columns:
    gap = table.copy()
    others = [c for c in table.columns if c!='stylegan_indist']
    for c in others: gap[f'gap_{c}'] = table['stylegan_indist'] - table[c]
    print('\nGeneralization gaps:'); print(gap.round(4).to_string())

ece = r[r['metric'].isin(['ECE','ECE_tempscaled'])]
if len(ece):
    ce = ece.pivot_table(index=['condition','test_gen'], columns='metric', values='value', aggfunc='last')
    print('\nCalibration (ECE before/after temperature):'); print(ce.round(4).to_string())
    (REPORTS_ROOT/'table_calibration.md').write_text(ce.round(4).to_markdown())
print('✅ Day 9 gate: master tables saved to reports/')